In [1]:
import requests 
from bs4 import BeautifulSoup as bs
import pandas as pd
import tldextract
import numpy as np
import matplotlib.pylab as plt
import os
import re

/home/jn/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def on_sale_chk(text):
    if len(text)<1:
        return False
    return 'domain' in text and 'sale' in text

def on_parked_chk(text):
    if len(text)<1:
        return True
    return 'domain' in text and 'park' in text

def on_Parked(text):
    if len(text)<1:
        return True
    return (('website' in text or 'content' in text) and 'unavailable' in text) or ('will' in text and 'soon' in text)

In [3]:
#returns html contents, textual character length, website size, status code, parked or on sale

def soupFromUrl(scrapeUrl):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
    try:
        req = requests.get(scrapeUrl, headers=headers, timeout=5)
        # print(req.status_code)
        req.close()
        if req.status_code == 200:
            # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
            soup = bs(req.text,'lxml')

            # print(soup)

            text = ''

            if soup.body:
                text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

            # print(soup)

            # print('text',text)
            # print(bs(req.text, 'html.parser'))
            # return [bs(req.text, 'html.parser'),len(req.text), len(req.content), req.status_code]
            # print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])
            return [len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))]
        else:
            # return [-1,0,0,req.status_code]
            return [0,0,req.status_code,0]
    except:
        # return [-1,0,0,-1]
        return [0,0,-1,0]

In [4]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 6.1; zh-CN) AppleWebKit/533+ (KHTML, like Gecko)'}
req = requests.get('https://www.lycos.com/', headers=headers, timeout=5)
print(req.status_code)
req.close()
if req.status_code == 200:
    # print(bs(req.text, 'html.parser').get_text().strip().replace('\n',' '))
    soup = bs(req.text,'lxml')

    # print(soup)
    text = ''

    if soup.body:
        text = re.sub(r'[^\w]', ' ',soup.body.get_text(' ', strip=True).lower())

    print(soup)
    print('text',text)
    print([len(req.content), len(text), req.status_code, 0+(on_sale_chk(text) or on_Parked(text) or on_parked_chk(text))])

200
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<!-- The above 3 meta tags *must* come first in the head; any other head content must come *after* these tags -->
<meta content="Lycos, Inc., is a web search engine and web portal established in 1994, spun out of Carnegie Mellon University. Lycos also encompasses a network of email, webhosting, social networking, and entertainment websites." name="description"/>
<meta content="" name="author"/>
<link href="https://ly.lygo.net/static/lycos/img/favicon.ico" rel="icon" type="image/png"/>
<title>Lycos.com</title>
<link href="//fonts.googleapis.com/css?family=Lato:400,300,300italic,400italic,700,700italic" rel="stylesheet" type="text/css"/>
<link href="/css/in/fonts.css" rel="stylesheet" type="text/css"/>
<link href="https://ly.lygo.net/static/lycos/css/in/font-awesome.css" rel="stylesheet" type="text

In [5]:
# print(soupFromUrl('https://www.delinian.com/'))
# print(soupFromUrl('https://www.makecashonline.com/'))
print(soupFromUrl('https://www.lycos.com/'))

[13597, 2569, 200, 0]


In [6]:
bangla_url = list(pd.read_csv('../Dataset/URL Data/bangla_spam.csv',delimiter='\t')['URL'])
bangla_url[:10]

['www.dot.gov.bd',
 'https://joinbangladesharmy.army.mil.bd',
 'www.hasinaandfriends.gov.bd',
 'https://joinbangladesharmy.mil.mil.bd',
 'https://joinairforce.baf.mil.bd',
 'www.joinnavy.navy.mil.bd',
 'http://cutt.ly/8CgjCP9',
 'http://joinnavy.navy.mil.bd',
 'www.bida.gov.bd',
 'www.nmst.gov.bd']

In [7]:
import re

def find_first_slash_preceded_by_number(s):
    # Regular expression to find the first instance of a number followed by '/'
    match = re.search(r'\d+/', s)
    
    if match:
        return match.start() + len(match.group()) - 1  # Return the index of '/'
    else:
        return -1  # Return -1 if no match is found

# Example usage
string = "example77/test 88/test2 99/test3"
index = find_first_slash_preceded_by_number(string)
print(index)  # Outputs the index of the first '/' preceded by a number

9


In [8]:
unique_bangla_url = set(bangla_url )

In [9]:
for idx,i in enumerate(unique_bangla_url):
    if '..' in i:
        if 'www' in i:
            unique_bangla_url[idx] = ''
        else:
            unique_bangla_url[idx] = unique_bangla_url[idx].replace('..','.')

In [10]:
unique_bangla_url = [i for i in unique_bangla_url if i!='']

In [11]:
bangla_dataset = {'ham':list(pd.read_csv('../Dataset/URL Data/bangla_spam_ham.csv',delimiter='\t')['URL']),'spam':list(pd.read_csv('../Dataset/URL Data/bangla_spam_spam.csv',delimiter='\t')['URL'])}

In [12]:
bangla_url_in_ham = [0]*len(unique_bangla_url)
bangla_url_in_spam = [0]*len(unique_bangla_url)

for idx,i in enumerate(unique_bangla_url):
    for j in bangla_dataset['ham']:
        if not isinstance(j, str):
            continue
        if i in j:
            bangla_url_in_ham[idx] = 1
            break
    for j in bangla_dataset['spam']:
        if not isinstance(j, str):
            continue
        if i in j:
            bangla_url_in_spam[idx] = 1
            break

print(bangla_url_in_ham.count(1))
print(bangla_url_in_spam.count(1))

43
48


In [13]:
common_urls = []
for i in range(len(bangla_url_in_ham)):
    if bangla_url_in_ham[i]==bangla_url_in_spam[i]:
        common_urls.append(unique_bangla_url[i])

len(common_urls)

3

In [14]:
bangla_dataset['Unique Url'] = unique_bangla_url

In [15]:
import tldextract

def FQDN(Url):
    
    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [16]:
bangla_dataset['FQDN'] = [FQDN(i) for i in bangla_dataset['Unique Url']]

len(set(bangla_dataset['FQDN']))

58

In [17]:
a = soupFromUrl('https://facebook.com')

print(a)

[78008, 32747, 200, 0]


The below query last ran on 5 October 2025

In [18]:
website_size, text_content_length, status_code, parked = [],[],[],[]

for i in bangla_dataset['FQDN']:
    a = soupFromUrl('https://'+i)

    website_size.append(a[0])
    text_content_length.append(a[1])
    status_code.append(a[2])
    parked.append(a[3])


# parked = [0]*len(bangla_dataset)

# for idx,i in enumerate(bangla_dataset['FQDN']):
#     if bangla_dataset['Status Code'][idx]==200:
#         a = soupFromUrl('https://'+i)
#         parked[idx] = a[3]
#         # break

# print(parked)

In [19]:
bangla_dataset['Website Size in KB'] = website_size
bangla_dataset['Website Textual Content Length'] = text_content_length
bangla_dataset['Status Code'] = status_code

bangla_dataset['Parked'] = parked

In [20]:
for i in bangla_dataset:
    print(len(bangla_dataset[i]))

90
89
88
88
88
88
88
88


In [21]:
bangla_dataset['ham'] = bangla_url_in_ham
bangla_dataset['spam'] = bangla_url_in_spam

In [22]:
bangla_dataset

{'ham': [0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  1],
 'spam': [1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0],
 'Unique Url': ['https://bKa.sh/8app',
  'https://rrj.nu/VucpYxnb',
  'https://click.daraz.com.bd/e/_C6

In [23]:
# bangla_dataset = pd.read_csv('../Dataset/URL Data/bangla Websites Analysis.csv')
bangla_dataset = pd.DataFrame.from_dict(bangla_dataset)
bangla_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,https://bKa.sh/8app,bKa.sh,1833,0,200,1
1,0,1,https://rrj.nu/VucpYxnb,rrj.nu,0,0,404,0
2,0,1,https://click.daraz.com.bd/e/_C69OCr,click.daraz.com.bd,559826,171763,200,1
3,0,1,https://rrj.nu/kFGj26Cd,rrj.nu,0,0,404,0
4,0,1,https://ebl.com.bd/Eid-Offers-2023#travel,ebl.com.bd,63664,6987,200,0


In [24]:
bangla_dataset['Status Code'].value_counts()

Status Code
 200    53
-1      30
 404     2
 403     1
 500     1
 530     1
Name: count, dtype: int64

In [26]:
numbers_to_replace = [501,403, 401]

# Value to replace with
new_value = 200

# Update the column
bangla_dataset.loc[bangla_dataset['Status Code'].isin(numbers_to_replace), 'Status Code'] = new_value

In [27]:
bangla_dataset['Status Code'].value_counts()

Status Code
 200    54
-1      30
 404     2
 500     1
 530     1
Name: count, dtype: int64

In [28]:
print(len(bangla_dataset[(bangla_dataset['Status Code']==200) & (bangla_dataset['ham']==1)]))
print(len(bangla_dataset[(bangla_dataset['Status Code']==200) & (bangla_dataset['spam']==1)]))

17
38


In [29]:
bangla_dataset['Parked'].value_counts()

Parked
0    65
1    23
Name: count, dtype: int64

In [30]:
print(len(bangla_dataset[(bangla_dataset['Parked']==1) & (bangla_dataset['ham']==1)]))
print(len(bangla_dataset[(bangla_dataset['Parked']==1) & (bangla_dataset['ham']==0)]))

3
20


In [32]:
bangla_dataset.head()

,ham,spam,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,0,1,https://bKa.sh/8app,bKa.sh,1833,0,200,1
1,0,1,https://rrj.nu/VucpYxnb,rrj.nu,0,0,404,0
2,0,1,https://click.daraz.com.bd/e/_C69OCr,click.daraz.com.bd,559826,171763,200,1
3,0,1,https://rrj.nu/kFGj26Cd,rrj.nu,0,0,404,0
4,0,1,https://ebl.com.bd/Eid-Offers-2023#travel,ebl.com.bd,63664,6987,200,0


In [33]:
bangla_dataset.to_csv('../Dataset/URL Data/bangla Websites Analysis.csv', index=None)

In [34]:
len(bangla_dataset)

88